In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Bakery daily revenue

Daily revenue ($) at a neighborhood bakery over 30 business days. A viral social media post on day 12 caused a spike. The shop closed for renovation on days 20–22.

No sub-questions. Use at least **five** of these: `diff()`, `pct_change()`, `ewm(span=6).mean()`, `expanding().mean()`, `np.nanmean`, `np.nanstd`, `np.nanpercentile`, `np.argmin`, `np.argmax`.

Answer two things in your print statements:
1. Which day had the biggest single-day drop in revenue (absolute terms)?
2. After the closure (day 23 onward), is the ewm revenue above or below the pre-viral average (days 1–11)?

In [12]:
bakery = pd.DataFrame({
    'day': pd.date_range('2024-03-01', periods=30, freq='B'),
    'revenue': [
        1250, 1320, 1180, 1290, 1350, 1210, 1280, 1340, 1260, 1310, 1390,  # days 1-11
        2840,                                                                # day 12 viral spike
        1890, 1650, 1540, 1480, 1420, 1390, 1460,                           # days 13-19
         180,  160,  200,                                                    # days 20-22 closure
        1310, 1380, 1420, 1460, 1510, 1480, 1530, 1490,                     # days 23-30
    ],
})

# Your code chere
bakery['abs_change'] = bakery['revenue'].diff()
bakery['rel_change'] = bakery['revenue'].pct_change()
bakery['cum_avg'] = bakery['revenue'].expanding().mean()
bakery['ewm_avg'] = bakery['revenue'].ewm(span = 6).mean()
max_rel_change_day = bakery.iloc[np.nanargmax(bakery['rel_change'])]['day'] 
max_abs_change_day = bakery.iloc[np.nanargmax(bakery['abs_change'])]['day'] 
max_abs_drop_day = bakery.iloc[np.nanargmin(bakery['abs_change'])]['day'] 
print(max_abs_drop_day,'has the largest drop')
mean_ewm1 = bakery.iloc[:11]['ewm_avg'].mean()
mean_ewm2 = bakery.iloc[:22]['ewm_avg'].mean()
print('pre ewm mean is: ', mean_ewm1)
print('post closure ewm mean is: ', mean_ewm2)
print('post is higher than pre viral')

2024-03-28 00:00:00 has the largest drop
pre ewm mean is:  1277.2648189384233
post closure ewm mean is:  1360.0838160448232
post is higher than pre viral


---

## Level 2 — Monthly app downloads

Monthly downloads for a productivity app over 24 months (Jan 2023 – Dec 2024). Major version releases at months 3, 10, and 18 caused download spikes.

1. `diff()` → `monthly_change`. Use `np.argsort` to find the 3 months with the largest gains and the 1 month with the biggest drop. Print them.
2. `ewm(span=4).mean()` → `ewm_dl` and `expanding().mean()` → `cum_avg`. At month 24, compare the two values — is the app accelerating or decelerating?
3. `np.nanpercentile` on `monthly_change` at [25, 50, 75]. What fraction of months had positive growth? Use `np.nanmean` on a boolean comparison.

In [40]:
apps = pd.DataFrame({
    'month': pd.date_range('2023-01', periods=24, freq='MS'),
    'downloads': [
        45000, 47200, 62400, 58100, 54800, 51200,
        53600, 56900, 60200, 78300, 73100, 68400,
        64800, 67200, 71500, 75900, 80200, 94600,
        88400, 83700, 87200, 91500, 95800, 99200,
    ],
})

# Your code here

apps['monthly_change'] = apps['downloads'].diff()
apps1 = apps.dropna()
s = np.argsort(-apps1['monthly_change'])
print('3 leargest gains are at: ', apps1.iloc[s.iloc[:3]]['month'])
print('1 leargest drop is at: ', apps1.iloc[s.iloc[-1]]['month'])

apps['ewm_dl'] = apps['downloads'].ewm(span = 4).mean()
apps['cum_avg'] = apps['downloads'].expanding().mean()
gap = apps.iloc[23]['ewm_dl'] - apps.iloc[23]['cum_avg'] 
print('since ewm is higher than cum average, the download is accelerating, the recent ones are higher')

qs = np.nanpercentile(apps['monthly_change'], q=[25,50,75])
print(np.nanmean(apps['monthly_change']>0),'had positive growth')

3 leargest gains are at:  9    2023-10-01
2    2023-03-01
17   2024-06-01
Name: month, dtype: datetime64[ns]
1 leargest drop is at:  2024-07-01 00:00:00
since ewm is higher than cum average, the download is accelerating, the recent ones are higher
0.625 had positive growth


---

## Level 3 — Streaming platforms

Weekly active subscribers (millions) for three streaming platforms over 16 weeks.

1. `groupby('platform').transform(lambda x: x.diff())` → `wow_sub`. Which platform × week had the biggest single-week subscriber loss? `np.nanstd` per platform — which is most volatile?
2. Add `ewm_sub` and `cum_avg` via `groupby().transform()`. At week 16, classify each platform as accelerating (ewm > cum_avg) or decelerating.
3. `np.corrcoef` on the three subscriber arrays — identify the most-correlated pair. Then build a plain Python list of platform names sorted by their week-16 `ewm_sub`, highest first. Which platform would you back as an investor, and why?

In [85]:
streams = pd.DataFrame({
    'week': list(pd.date_range('2024-01-07', periods=16, freq='W')) * 3,
    'platform': ['StreamNow']*16 + ['FlixPlus']*16 + ['ViewMax']*16,
    'subscribers': [
         8.2,  8.5,  8.3,  8.7,  8.5,  8.9,  8.7,  9.1,
         8.9,  9.3,  9.2,  9.6,  9.4,  9.8,  9.7, 10.1,   # StreamNow
        12.1, 13.5, 12.3, 13.8, 12.6, 14.1, 12.9, 14.4,
        13.2, 14.7, 13.5, 15.0, 13.8, 15.3, 14.1, 15.6,   # FlixPlus
        15.8, 15.2, 15.6, 14.9, 15.1, 14.5, 14.8, 14.2,
        14.5, 13.9, 14.1, 13.6, 13.8, 13.3, 13.5, 13.1,   # ViewMax
    ],
})

# Your code here
streams['wow_sub'] = streams.groupby('platform')['subscribers'].transform(lambda x: x.diff())

print(streams.loc[streams['wow_sub'].idxmin(),'week'],streams.loc[streams['wow_sub'].idxmin(),'platform'], 'had the biggest single week subscriber loss' )
pp = streams.groupby('platform')['wow_sub'].apply(lambda x: np.nanstd(x))
print(pp.idxmax(), 'is th emost volatile')

streams['ewm_sub'] = streams.groupby('platform')['subscribers'].transform(lambda x: x.ewm(span = 5).mean())
streams['cum_sub'] = streams.groupby('platform')['subscribers'].transform(lambda x: x.expanding().mean())

w16 = streams['week'].unique()[-1]
s16 = streams.loc[streams['week'] == w16].copy()
s16['gap'] = s16['ewm_sub'] - s16['cum_sub']
print(s16['gap'])
print('Steram now and Flixplus are increasing, VeiwMax is decreasing')


sn = ['StreamNow','FlixPlus','ViewMax']
s0 = streams.loc[streams['platform']== sn[0],'subscribers']
s1 = streams.loc[streams['platform']== sn[1],'subscribers']
s2 = streams.loc[streams['platform']== sn[2],'subscribers']

sc = np.corrcoef([s0, s1, s2])
np.fill_diagonal(sc, -np.inf)
m = np.argmax(sc)
m = np.unravel_index(m, shape = sc.shape)
print('most correlated pair is ', sn[m[0]], sn[m[1]])

pp = s16.iloc[np.argsort(-s16['ewm_sub'])]['platform']
pp = pp.to_list()
print('I woould be the investor of: ', pp[0],'because it is accelerating')

2024-02-04 00:00:00 FlixPlus had the biggest single week subscriber loss
FlixPlus is th emost volatile
15    0.673266
31    0.957486
47   -0.894778
Name: gap, dtype: float64
Steram now and Flixplus are increasing, VeiwMax is decreasing
most correlated pair is  StreamNow FlixPlus
I woould be the investor of:  FlixPlus because it is accelerating


In [84]:
s16

,week,platform,subscribers,wow_sub,ewm_sub,cum_sub,gap
15,2024-04-21,StreamNow,10.1,0.4,9.729516,9.05625,0.673266
31,2024-04-21,FlixPlus,15.6,1.5,14.763736,13.80625,0.957486
47,2024-04-21,ViewMax,13.1,-0.4,13.473972,14.36875,-0.894778
